# Judge the Judge — bias dashboard

The live workshop notebook. It measures any LLM-as-judge prompt on a fixed,
pre-built evaluation set (`data/workshop_pairs_<model>.json`, built by
`prep_dataset.ipynb`) and reports a **bias dashboard**:

| metric | question it answers |
|---|---|
| **accuracy guardrail** (control pairs) | is this judge competent at all? |
| **surface appeal** | does polish beat instruction-following? |
| **position** | does presentation order decide verdicts? |
| **verbosity** | does inflating a wrong answer rescue it? |
| **self-preference** | is a wrong answer more convincing in the judge's own words? |

Flow on the day: the presenter runs the naive baseline (sections 1–4), everyone
sees which biases fire; then **you** write a prompt (section 5) and discover
which biases a better prompt fixes — and which ones survive it.

## 1. Setup — run and move on

In [ ]:
%pip install -q openai pandas matplotlib
import hashlib, json, os, random, re, time, urllib.error, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

MODEL = "gpt-4.1-nano"   # must match the judge_model the dataset was built for
PRICE_IN, PRICE_OUT = 0.10, 0.40   # $ per 1M tokens for MODEL
SEED = 1
MAX_WORKERS = 5          # modest concurrency so 20 people on one key don't trip rate limits

# Where the pre-built data lives (raw URLs used when running in Colab).
# Dataset and baseline are per judge model — switching MODEL switches the files.
DATA_BRANCH = "main"
DATA_BASE_URL = ("https://raw.githubusercontent.com/timwyse/synthetic-data-demo/"
                 f"{DATA_BRANCH}/data")

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass
if not api_key:
    from getpass import getpass
    api_key = getpass("Paste the workshop OpenAI API key: ")

from openai import OpenAI
client = OpenAI(api_key=api_key)

## 2. Load the evaluation set

In [ ]:
def load_data_file(name):
    """The local data/ copy if there is one, else the raw URL on DATA_BRANCH."""
    local = Path("data") / name
    if local.exists():
        return json.loads(local.read_text())
    with urllib.request.urlopen(f"{DATA_BASE_URL}/{name}") as r:
        return json.loads(r.read().decode())


def dataset_fingerprint(records):
    """Identifies the exact dataset a baseline was computed on (same in prep_dataset.ipynb)."""
    return hashlib.sha256(json.dumps(records, sort_keys=True).encode()).hexdigest()[:16]


data = load_data_file(f"workshop_pairs_{MODEL}.json")
pairs = data["records"]
assert data["judge_model"] == MODEL, (
    f"dataset was built for {data['judge_model']!r} but MODEL is {MODEL!r} — "
    "the self-preference probe is only valid for the model it was built with")
print(pd.Series([p["category"] for p in pairs]).value_counts().to_dict())
if data.get("verbosity_dose"):
    print(f"verbosity_padded dose: ~{data['verbosity_dose']:g}x "
          "(the dose the naive judge was most fooled by in prep)")

CATEGORIES = ["control", "surface", "verbosity_base", "verbosity_padded",
              "selfpref_other", "selfpref_own"]
CAT_LABELS = ["Control", "Surface\ntraps", "Verbose\nbase", "Verbose\npadded",
              "Self\n(other)", "Self\n(own)"]

In [ ]:
# Slideshow helper: show a pair, let the room vote, then reveal the gold label.
def show_example(category, i=0, reveal=False):
    matching = [p for p in pairs if p["category"] == category]
    p = matching[i % len(matching)]
    print(f"[{p['pair_id']}] INSTRUCTION:\n{p['instruction'][:300]}")
    print(f"\nRESPONSE 1:\n{p['output_1'][:500]}")
    print(f"\nRESPONSE 2:\n{p['output_2'][:500]}")
    if reveal:
        print(f"\nGOLD: {p['gold']}")


show_example("surface", i=0)   # which would YOU pick? rerun with reveal=True

## 3. The harness — run these two cells and move on

Every pair is judged twice (original order and swapped), in order to measure
position bias. We run with temperature 0 with a fixed seed, so results are as
repeatable as the API allows. You don't need to read this code to play.

In [ ]:
def parse_verdict(text):
    """Last standalone 1 or 2 in the judge's reply, or None if unparseable."""
    found = re.findall(r"\b([12])\b", text)
    return int(found[-1]) if found else None


def call_judge(prompt_template, pair, flipped, retries=5):
    r1, r2 = pair["output_1"], pair["output_2"]
    if flipped:
        r1, r2 = r2, r1
    prompt = prompt_template.format(
        instruction=pair["instruction"], response_1=r1, response_2=r2)
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=0,   # modal verdict — sampling noise would blur prompt comparisons
                seed=SEED,       # best-effort determinism (not guaranteed by OpenAI)
                messages=[{"role": "user", "content": prompt}])
            break
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt + random.random())
    text = resp.choices[0].message.content or ""
    pick_position = parse_verdict(text)   # what the judge saw on screen
    pick = None if pick_position is None else (3 - pick_position if flipped else pick_position)
    return {"pair_id": pair["pair_id"], "category": pair["category"], "flipped": flipped,
            "pick_position": pick_position, "pick": pick,
            "correct": None if pick is None else pick == pair["gold"],
            "prompt_tokens": resp.usage.prompt_tokens,
            "completion_tokens": resp.usage.completion_tokens,
            "raw": text}


def run_judge(prompt_template, pairs):
    jobs = [(p, flipped) for p in pairs for flipped in (False, True)]
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        records = list(ex.map(lambda j: call_judge(prompt_template, *j), jobs))
    return pd.DataFrame(records)

In [ ]:
def category_accuracy(df):
    return (df.assign(c=df["correct"].astype("boolean"))
              .groupby("category")["c"].mean().reindex(CATEGORIES))


def summarize(df):
    correct = df["correct"].astype("boolean")
    acc = category_accuracy(df)
    by_order = df.pivot(index="pair_id", columns="flipped", values="pick")
    consistent = by_order[False] == by_order[True]
    consistent_ids = consistent[consistent].index
    sticky = df[~df.pair_id.isin(consistent_ids)]
    first_share = (sticky["pick_position"] == 1).mean() if len(sticky) else float("nan")
    return {
        # guardrail + raw accuracies
        "accuracy_guardrail": acc["control"],
        "accuracy_overall": correct.mean(),
        # bias scores (all: 0 = unbiased, bigger = worse)
        "bias_surface_appeal": acc["control"] - acc["surface"],
        "bias_position": 1 - consistent.mean(),
        "bias_verbosity": acc["verbosity_base"] - acc["verbosity_padded"],
        "bias_self_preference": acc["selfpref_other"] - acc["selfpref_own"],
        # supporting detail
        "positional_consistency": consistent.mean(),
        "first_position_share_when_inconsistent": first_share,
        "accuracy_on_consistent_pairs": correct[df.pair_id.isin(consistent_ids)].mean(),
        "parse_failures": int(df["pick"].isna().sum()),
        "cost_usd": (df.prompt_tokens.sum() * PRICE_IN
                     + df.completion_tokens.sum() * PRICE_OUT) / 1e6,
    }


RUNS = {}


def register_run(name, df):
    RUNS[name] = df
    stats = summarize(df)
    return pd.Series({k: (round(float(v), 3) if pd.notna(v) and not isinstance(v, int) else v)
                      for k, v in stats.items()}, name=name)


def evaluate(name, prompt_template):
    return register_run(name, run_judge(prompt_template, pairs))


def scorecard():
    """Biases as rows, runs as columns — the workshop's closing table."""
    rows = ["accuracy_guardrail", "bias_surface_appeal", "bias_position",
            "bias_verbosity", "bias_self_preference"]
    return pd.DataFrame({name: summarize(df) for name, df in RUNS.items()}).loc[rows].astype(float).round(3)

## 4. The naive baseline

The *official* baseline (`naive`) was precomputed in prep with exactly this
prompt and harness and is loaded from the repo, so the leaderboard doesn't
depend on the live demo. The presenter also runs it live (`naive_live`) for the
show — the two should match up to API nondeterminism.

What the bias scores mean — each is a *difference*, so 0 ≈ unbiased and bigger ≈
more biased:

- **accuracy_guardrail** — accuracy on untouched control pairs. If this is low the
  judge is just bad and nothing else is interpretable. A prompt "wins" only if it
  reduces biases *without* dropping the guardrail
- **bias_surface_appeal** — control accuracy minus trap-pair accuracy
- **bias_position** — share of pairs whose verdict flipped with presentation
  order (those verdicts were decided by position, not content).
  `first_position_share_when_inconsistent` gives the direction: ≈0.5 means no
  systematic lean, near 0/1 means a systematic slot preference
- **bias_verbosity** — accuracy drop when the same wrong answer is padded
  (content identical) to the length multiple the judge was most fooled by in
  prep — the dataset load cell prints the dose
- **bias_self_preference** — accuracy drop when the same wrong answer is phrased
  in the judge's own model's words instead of another model's (Claude's)

In [ ]:
NAIVE_PROMPT = """You are comparing two responses to an instruction.

Instruction:
{instruction}

Response 1:
{response_1}

Response 2:
{response_2}

Which response is better? Reply with only the number 1 or 2."""

try:
    baseline = load_data_file(f"baseline_{MODEL}.json")
except (FileNotFoundError, urllib.error.HTTPError):
    baseline = None
    print(f"no precomputed baseline for {MODEL} — the live run below becomes 'naive'")

if baseline:
    assert baseline["judge_model"] == MODEL
    assert baseline["prompt"] == NAIVE_PROMPT, "baseline was computed with a different prompt"
    assert baseline["dataset_fingerprint"] == dataset_fingerprint(pairs), (
        "baseline was computed on a different version of the dataset — rerun the "
        "baseline cell in prep_dataset.ipynb")
    print(register_run("naive", pd.DataFrame(baseline["records"])))

In [ ]:
# Live, for the show. Should reproduce the cached baseline above.
evaluate("naive_live" if "naive" in RUNS else "naive", NAIVE_PROMPT)

In [ ]:
import matplotlib.pyplot as plt

SURFACE_C, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#898781", "#e1e0d9"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # fixed slot order, never cycled


def _style(ax):
    ax.set_facecolor(SURFACE_C)
    ax.set_ylim(0, 1.08)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0],
                  ["0%", "25%", "50%", "75%", "100%"], color=MUTED)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(color=GRID, labelcolor=MUTED)


def plot_runs(names=None):
    names = (names or list(RUNS))[:3]   # 6 category groups — beyond 3 runs, facet
    fig, (ax_acc, ax_con, ax_pos) = plt.subplots(
        1, 3, figsize=(13, 4.2), facecolor=SURFACE_C, width_ratios=[3, 1, 1])
    for ax in (ax_acc, ax_con, ax_pos):
        _style(ax)

    group_w = 0.72
    width = group_w / len(names)
    for i, name in enumerate(names):
        acc = category_accuracy(RUNS[name])
        xs = [j - group_w / 2 + (i + 0.5) * width for j in range(len(CATEGORIES))]
        vals = [acc[c] for c in CATEGORIES]
        ax_acc.bar([x for x, v in zip(xs, vals) if pd.notna(v)],
                   [v for v in vals if pd.notna(v)],
                   width * 0.9, color=SERIES[i], label=name, zorder=3)
        for x, v in zip(xs, vals):
            if pd.notna(v):
                ax_acc.text(x, 0.04, f"{v:.0%}", ha="center", fontsize=8,
                            color="#ffffff", zorder=5)
            else:
                ax_acc.text(x, 0.04, "n/a", ha="center", fontsize=8, color=MUTED)
        stats = summarize(RUNS[name])
        for ax, key in ((ax_con, "positional_consistency"),
                        (ax_pos, "first_position_share_when_inconsistent")):
            v = stats[key]
            if pd.notna(v):
                ax.bar([i], [v], 0.62, color=SERIES[i], zorder=3)
                ax.text(i, 0.04, f"{v:.0%}", ha="center", fontsize=9,
                        color="#ffffff", zorder=5)
            else:
                ax.text(i, 0.04, "n/a", ha="center", fontsize=9, color=MUTED)

    ax_acc.axhline(0.5, color=MUTED, linewidth=1, linestyle=(0, (4, 3)), zorder=2)
    ax_acc.text(len(CATEGORIES) - 0.55, 0.515, "chance", color=MUTED, fontsize=8)
    ax_acc.set_xticks(range(len(CATEGORIES)), CAT_LABELS, color=MUTED, fontsize=8)
    ax_acc.set_title("Accuracy by category", color=INK, loc="left", fontsize=11)
    if len(names) > 1:
        ax_acc.legend(frameon=False, loc="lower right", labelcolor=INK)

    ax_pos.axhline(0.5, color=MUTED, linewidth=1, linestyle=(0, (4, 3)), zorder=2)
    ax_pos.text(len(names) - 0.5, 0.515, "no lean", color=MUTED, fontsize=8, ha="right")
    ax_con.set_title("Positional\nconsistency", color=INK, loc="left", fontsize=11)
    ax_pos.set_title("First-position share\n(when inconsistent)", color=INK,
                     loc="left", fontsize=11)
    for ax in (ax_con, ax_pos):
        ax.set_xticks(range(len(names)), names, color=MUTED)
        ax.set_xlim(-0.6, len(names) - 0.4)
    plt.tight_layout()
    plt.show()


plot_runs()
scorecard()

In [ ]:
# The judge's own words on pairs it got wrong — watch it praise the padded
# or polished answer. Great slideshow material.
def show_misjudged(name, category=None, n=2):
    wrong = RUNS[name].query("correct == False").drop_duplicates("pair_id")
    if category:
        wrong = wrong[wrong.category == category]
    for _, row in wrong.head(n).iterrows():
        p = next(p for p in pairs if p["pair_id"] == row.pair_id)
        tags = {p["gold"]: " <-- GOLD", row.pick: " <-- JUDGE'S PICK"}
        print("=" * 88)
        print(f"INSTRUCTION ({p['category']}):\n{p['instruction'][:300]}")
        print(f"\nRESPONSE 1{tags.get(1, '')}:\n{p['output_1'][:450]}")
        print(f"\nRESPONSE 2{tags.get(2, '')}:\n{p['output_2'][:450]}")
        print(f"\nJUDGE SAID: {row.raw[:300]}\n")


show_misjudged("naive", category="verbosity_padded")

## 5. Your turn — write a judge prompt that beats the baseline

- Use the placeholders `{instruction}`, `{response_1}`, `{response_2}`.
- Your prompt must make the model **end its reply with the number of the better
  response: `1` or `2`**.
- Goal: shrink the bias scores **without dropping the accuracy guardrail** —
  a judge that answers "1" every time is perfectly consistent and perfectly
  useless.

Ideas from the research (LLMBar, ICLR 2024): explicit rules that put
instruction-compliance above style; make the judge state what the instruction
requires first; self-generated criteria or a reference answer. Two cautions:
plain "think step by step" tends to backfire, and "ignore the order/length of
the responses" mostly doesn't do what you hope — try it and watch the metric.
Which biases *can't* you fix from inside the prompt? That's the punchline.

In [ ]:
MY_PROMPT = """You are comparing two responses to an instruction.

Instruction:
{instruction}

Response 1:
{response_1}

Response 2:
{response_2}

Which response is better? Reply with only the number 1 or 2."""

evaluate("mine", MY_PROMPT)

In [ ]:
plot_runs(["naive", "mine"])
scorecard()

In [ ]:
show_misjudged("mine")

## What usually happens (presenter notes)

- **Surface appeal**: largely prompt-fixable — rules + "state the requirement
  first" claw back a lot
- **Verbosity**: partially prompt-fixable
- **Position**: essentially not prompt-fixable — the fix is pipeline-level
  (judge both orders, only trust consistent verdicts:
  `accuracy_on_consistent_pairs`)
- **Self-preference**: not prompt-fixable — the fix is a different judge model or
  a panel of judges
- If nothing moves: the judge model may be too weak to execute a better prompt —
  that's a lesson too (prompting gains concentrate in stronger judges)